# The Supply Line

**An Agentic AI Exercise for Business Professionals**

You're an operations coordinator at NovaTrade, a Swiss logistics company. Your order queue has incoming requests — reorders, shipping delays, quality rejections, customs holds, supplier disputes, rush orders, and system alerts. Your job: fulfill as many orders as possible while keeping quality high.

**Win Condition:** Fulfill 15+ orders with Quality >= 80% without losing all your operations tokens.

**Rules:**
- You have **3 operations tokens**. Wrong or unnecessary escalations cost a token. Lose all 3 = game over.
- Ordering from a **blacklisted supplier** costs a token.
- Pricing disputes >5% **must** be escalated to procurement.
- Orders >EUR 50,000 **must** be escalated to finance.
- Rush orders under 1000 units can use expedited shipping — do NOT escalate these.
- There are **two boss fights**:
  - **Project Aurora** (T-001): Requires resolving 3 prerequisite orders, then `prepare_launch_briefing()`, then `authorize_launch()`.
  - **STA Regulatory Review** (T-020): Requires resolving 3 prerequisite orders, then `prepare_compliance_package()`, then `submit_compliance()`.
- System alerts are informational — just acknowledge and close them.

First, **play the game yourself** to understand the mechanics. Then implement an AI agent to do it for you.

In [ ]:
# Setup
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists('coding-exercises'):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir('coding-exercises/the_supply_line')

sys.path.insert(0, '.')

!pip install google-genai --quiet
print('Setup complete!')

In [ ]:
from supply_line import *
print('The Supply Line loaded!')

# Preview the queue
agent, world, tools = create_game()
print()
print(world.queue_summary())

---
## Play the Game Yourself!

Use the interactive controls below to process orders. Tips:

1. **Read an order** from the queue (highest priority first)
2. **Look up** the knowledge base before taking action
3. **Apply the correct action** (place_order, submit_documents, reject_shipment, etc.)
4. **Notify the client** using the appropriate template
5. **Escalate** only when the KB says you should — wrong escalations cost tokens!
6. **Close** the order when you're done

Watch your Quality score — closing without looking things up or notifying clients hurts your rating.

**Boss fights:** Project Aurora (T-001) and the STA Regulatory Review (T-020) require resolving prerequisite orders first. Think about **dependency resolution** — which orders must be done before others?

In [ ]:
from supply_line.interactive import play_interactive

game = play_interactive()

---
## TODO: LLM-Powered Think Function

Now automate it! Use an LLM to make decisions. But keep these principles in mind:

1. **Autopilot** — handle obvious orders (system alerts, simple reorders) without the LLM
2. **Deterministic scoring** — prioritize orders by formula, not LLM judgment
3. **Memory** — summarize order state, don't dump raw history
4. **Guardrails** — validate LLM output before executing (block blacklisted suppliers, unverified escalations)
5. **Prompt engineering** — give the LLM clear tools, examples, and constraints

The function receives:
- `agent`: your current state (tokens, resolved count, quality score)
- `world`: the supply world (order queue, knowledge base, entities)
- `history`: list of previous observations, actions, and results

It must return a string like: `'ACTION: read_order(order_id="T-001")'`

**Available actions:**
- `check_orders()` — see the queue (free)
- `read_order(order_id="...")` — open an order
- `lookup_kb(query="...")` — search the knowledge base
- `check_inventory(sku="...")` — check stock levels
- `lookup_shipment(shipment_id="...")` — get shipment status
- `lookup_contract(supplier="...")` — get contract terms
- `check_documentation(shipment_id="...")` — check customs docs
- `check_alternatives()` — find alternative suppliers/carriers
- `place_order(supplier="...", sku="...", quantity="...", expedited="false")` — place a purchase order
- `reject_shipment(shipment_id="...", reason="...")` — reject a QC-failed shipment
- `file_claim(supplier="...", amount="...")` — file a claim
- `submit_documents(shipment_id="...", doc_type="...")` — submit customs docs
- `update_timeline(order_id="...", new_eta="...")` — update delivery timeline
- `verify_pricing(order_id="...")` — verify pricing against contract
- `compile_records(period="...")` — compile import records for audit
- `acknowledge_alert(alert_id="...")` — acknowledge a system alert
- `notify_client(template="...")` — send notification using a template
- `escalate(department="...", reason="...")` — escalate to a department
- `prepare_launch_briefing()` — compile Project Aurora launch briefing
- `authorize_launch(order_id="T-001")` — authorize the launch
- `prepare_compliance_package()` — compile STA compliance package
- `submit_compliance(order_id="T-020")` — submit to STA
- `close_order(order_id="...")` — mark order as fulfilled (scores quality)
- `check_stats()` — see your stats (free)
- `take_break()` — recover 1 token (costs 5 turns)

**Strategy hints:**
- Process orders by priority (critical first) — but beware: boss orders have **dependencies**!
- Always lookup KB before handling unfamiliar order types
- System alerts: just `acknowledge_alert` — they don't affect quality
- Quality rejections need: reject_shipment, file_claim, place_order (alternative supplier), notify
- Customs holds need: check_documentation, submit_documents, notify
- Rush orders under 1000 units: use expedited shipping, do NOT escalate
- Pricing disputes >5%: MUST escalate to procurement
- The **two boss fights** (Aurora T-001 and STA T-020) each need 3 prerequisite orders resolved first. Think about **dependency resolution** — skip boss orders until their prerequisites are done!

In [ ]:
import os
from google import genai

# os.environ['GEMINI_API_KEY'] = 'your-key-here'
try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
except (ImportError, Exception):
    api_key = os.environ.get('GEMINI_API_KEY')

client = genai.Client(api_key=api_key)
print('Gemini client ready!')

In [ ]:
def think_llm(agent: AgentState, world: SupplyWorld, history: list[dict]) -> str:
    """Decide the next action using an LLM.
    
    Args:
        agent: Current agent state (tokens, resolved count, quality).
        world: The supply world (queue, orders, KB, entities).
        history: List of dicts with observations, actions, results.
    
    Returns:
        str: An ACTION: call string.
    """
    # ========================
    # YOUR CODE HERE
    # ========================
    raise NotImplementedError('TODO: Implement think_llm')

In [ ]:
result = play_with_llm(
    think_fn=lambda agent, world, history: think_llm(agent, world, history),
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Not quite...'} | "
      f"Resolved: {result['resolved']} | Quality: {result['quality']}% | "
      f"Tokens: {result['tokens']} | Turns: {result['turns']}")
print(f"Log: {result['log_file']}")

In [ ]:
# Download the game log
try:
    from google.colab import files
    files.download(result['log_file'])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print('Open the file to inspect your agent\'s decisions turn by turn.')